# 🎲 Module 1.4 — Measuring Risk II: Scenarios & Distributions
### *From Theory to Real-World Trading with Python*

---

**Course:** Capital Markets Recap — Level 1: Foundations  
**Prerequisites:** Modules 1.1–1.3  
**Tools:** `yfinance`, `pandas`, `numpy`, `matplotlib`, `scipy`

---

## What This Module Covers

Module 1.3 treated volatility as if returns were normally distributed. This module confronts that assumption head-on. Real financial returns are **not normal** — they have:
- **Skewness** — more extreme negative returns than positive ones
- **Fat tails (leptokurtosis)** — crashes happen far more often than a normal distribution predicts
- **Scenario dependence** — your expected return depends heavily on *which market regime* you're in

Your lecture notes covered scenarios and probability directly:
> *"Inversión con 10% de retorno esperado con 2 escenarios: 50% → -10%, 50% → 30%"*

This module extends that intuition to real market data:

1. **Return distributions in depth** — skewness, kurtosis, fat tails
2. **Scenario analysis** — bull, bear, and normal market regimes
3. **Expected Shortfall (CVaR)** — what happens *beyond* the VaR threshold
4. **Monte Carlo simulation** — projecting future returns under uncertainty
5. **Stress testing** — what would a 2008-style crash do to your portfolio?

---
## 0. Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import norm, t as t_dist
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # reproducibility for Monte Carlo

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

TRADING_DAYS = 252
START = '2005-01-01'   # Longer history to capture more regime variation
END   = '2024-12-31'

print("Setup complete ✅")

In [ ]:
# ── Download data ─────────────────────────────────────────────────────────────
# Longer window than previous modules to include 2008 GFC

tickers = ['SPY', 'AAPL', 'GLD', '^VIX']
raw = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)['Close']
raw.columns = ['AAPL', 'GLD', 'SPY', 'VIX']

spy_ret  = raw['SPY'].pct_change().dropna()
aapl_ret = raw['AAPL'].pct_change().dropna()
gld_ret  = raw['GLD'].pct_change().dropna()

spy_log  = np.log(raw['SPY'] / raw['SPY'].shift(1)).dropna()

print(f"Downloaded {len(spy_ret)} trading days ({START[:4]}–{END[:4]})")
print(f"Date range: {spy_ret.index[0].date()} → {spy_ret.index[-1].date()}")

---
## 1. The Shape of Return Distributions

### Theory

A **normal distribution** is fully described by two parameters: mean ($\mu$) and standard deviation ($\sigma$). But real return distributions are characterized by **four moments**:

| Moment | Parameter | Normal value | Finance interpretation |
|---|---|---|---|
| 1st | Mean $\mu$ | Any value | Expected return |
| 2nd | Variance $\sigma^2$ | Any positive | Volatility (risk) |
| 3rd | **Skewness** | **0** | Asymmetry of returns |
| 4th | **Excess Kurtosis** | **0** | Fat-tailedness |

$$
\text{Skewness} = \frac{E[(R - \mu)^3]}{\sigma^3} \qquad \text{Excess Kurtosis} = \frac{E[(R - \mu)^4]}{\sigma^4} - 3
$$

**Skewness in financial returns:**
- Stocks tend to have **negative skew** — small gains most of the time, occasional large crashes
- This means the *median* return > *mean* return (the mean is dragged down by rare catastrophic losses)

**Excess Kurtosis (fat tails):**
- Stocks have **positive excess kurtosis (leptokurtosis)** — tails are fatter than normal
- Events that the normal distribution says should happen once in 10,000 years actually happen once per decade

> 💡 **Why this matters for trading:** A strategy that looks great based on mean and volatility alone can be catastrophically wrong if it ignores skewness and kurtosis. The 2008 financial crisis destroyed many 'low-risk' strategies that had been modeled with Gaussian returns.

In [ ]:
# ── 1a. Compute the four moments for our assets ───────────────────────────────

def return_moments(ret_series, name):
    """Compute and display the four moments of a return series."""
    mu      = ret_series.mean() * TRADING_DAYS * 100          # annualized mean
    sigma   = ret_series.std() * np.sqrt(TRADING_DAYS) * 100  # annualized vol
    skew    = stats.skew(ret_series)
    kurt    = stats.kurtosis(ret_series)                       # excess kurtosis
    jb_p    = stats.jarque_bera(ret_series)[1]
    return {'Asset': name, 'Ann. Mean %': round(mu, 2), 'Ann. Vol %': round(sigma, 2),
            'Skewness': round(skew, 3), 'Excess Kurtosis': round(kurt, 3),
            'JB p-value': f'{jb_p:.2e}'}

moments_df = pd.DataFrame([
    return_moments(spy_ret,  'SPY (S&P 500)'),
    return_moments(aapl_ret, 'AAPL'),
    return_moments(gld_ret,  'GLD (Gold)'),
]).set_index('Asset')

print("Return Distribution Moments:")
print(moments_df.to_string())
print("\nNormal distribution reference: Skewness=0, Excess Kurtosis=0, JB p→1")
print("JB p-value << 0.05 → REJECT normality → returns are NOT normally distributed")

In [ ]:
# ── 1b. Visualize the departure from normality ────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

assets = [('SPY', spy_ret, '#2980b9'), ('AAPL', aapl_ret, '#27ae60'), ('GLD', gld_ret, '#f39c12')]

for i, (name, ret, color) in enumerate(assets):
    mu_d, sig_d = ret.mean(), ret.std()
    skew_val = stats.skew(ret)
    kurt_val = stats.kurtosis(ret)

    # Top row: Distribution histogram
    ax_hist = fig.add_subplot(gs[0, i])
    ax_hist.hist(ret * 100, bins=100, density=True, color=color, alpha=0.5)
    x = np.linspace(ret.min(), ret.max(), 400)
    ax_hist.plot(x * 100, norm.pdf(x, mu_d, sig_d) / 100,
                 'k-', linewidth=2, label='Normal fit')

    # Shade the tails
    var99 = mu_d - 2.326 * sig_d
    tail_mask = x < var99
    ax_hist.fill_between(x[tail_mask] * 100,
                         norm.pdf(x[tail_mask], mu_d, sig_d) / 100,
                         alpha=0.4, color='red', label='Normal 1% tail')
    ax_hist.set_title(f'{name}\nSkew={skew_val:.2f}, Kurt={kurt_val:.2f}',
                      fontweight='bold', fontsize=10)
    ax_hist.set_xlabel('Daily Return (%)')
    if i == 0:
        ax_hist.set_ylabel('Density')
    ax_hist.legend(fontsize=8)

    # Bottom row: Q-Q plot
    ax_qq = fig.add_subplot(gs[1, i])
    (osm, osr), (slope, intercept, _) = stats.probplot(ret, dist='norm')
    ax_qq.scatter(osm, osr, color=color, s=2, alpha=0.5)
    line_x = np.array([min(osm), max(osm)])
    ax_qq.plot(line_x, slope * line_x + intercept, 'r-', linewidth=1.5,
               label='Normal line')
    ax_qq.set_title(f'{name} — Q-Q Plot', fontweight='bold', fontsize=10)
    ax_qq.set_xlabel('Theoretical Quantiles')
    if i == 0:
        ax_qq.set_ylabel('Sample Quantiles')
    ax_qq.legend(fontsize=8)

plt.suptitle('Return Distributions vs. Normal — Histograms and Q-Q Plots',
             fontweight='bold', fontsize=13)
plt.show()
print("\nQ-Q plot: Deviations from the red line in the tails = fat tails")
print("The S-curve pattern is the signature of leptokurtosis (excess kurtosis)")

---
## 2. Scenario Analysis — Bull, Bear, and Normal Regimes

### Theory

Your lecture notes worked through discrete scenarios with assigned probabilities:
> *"Inversión con 10% de retorno esperado: Escenario 1 (50%) → 45%; Escenario 2 (50%) → -25%"*

In real markets, this translates to **market regimes**:

| Regime | Characteristics | How to identify |
|---|---|---|
| **Bull market** | Rising prices, low volatility, positive news | VIX < 20, returns above 12-month average |
| **Bear market** | Falling prices (>20% decline), high volatility | VIX > 25, S&P 500 -20% from peak |
| **Crisis** | Sharp crash, extreme volatility, correlation spike | VIX > 35, broad asset class declines |

**Why scenario analysis matters for trading:**
- Your portfolio may perform well on average but fail catastrophically in one specific scenario
- Stress tests reveal hidden concentrations and correlated risk
- Scenario probabilities let you calculate a **probability-weighted expected return** — the core of expected value decision-making

$$
E[R] = \sum_{i} p_i \cdot R_i
$$

In [ ]:
# ── 2a. Define market regimes using VIX ──────────────────────────────────────
vix = raw['VIX'].dropna()

# Align VIX with SPY returns
common_idx = spy_ret.index.intersection(vix.index)
spy_aligned = spy_ret.loc[common_idx]
vix_aligned = vix.loc[common_idx].shift(1).dropna()  # use previous day's VIX
spy_aligned = spy_aligned.loc[vix_aligned.index]

# Classify regimes
def classify_regime(vix_level):
    if vix_level < 15:
        return 'Bull (VIX < 15)'
    elif vix_level < 25:
        return 'Normal (15-25)'
    elif vix_level < 35:
        return 'Elevated (25-35)'
    else:
        return 'Crisis (VIX > 35)'

regimes = vix_aligned.apply(classify_regime)

# Calculate stats by regime
regime_stats = pd.DataFrame()
for regime_name in ['Bull (VIX < 15)', 'Normal (15-25)', 'Elevated (25-35)', 'Crisis (VIX > 35)']:
    mask = regimes == regime_name
    ret_in_regime = spy_aligned[mask]
    if len(ret_in_regime) > 10:
        regime_stats.loc[regime_name, 'Days']         = len(ret_in_regime)
        regime_stats.loc[regime_name, 'Freq (%)']     = round(mask.mean() * 100, 1)
        regime_stats.loc[regime_name, 'Avg Daily Ret (%)'] = round(ret_in_regime.mean() * 100, 4)
        regime_stats.loc[regime_name, 'Ann. Return (%)']  = round(ret_in_regime.mean() * TRADING_DAYS * 100, 1)
        regime_stats.loc[regime_name, 'Ann. Vol (%)'] = round(ret_in_regime.std() * np.sqrt(TRADING_DAYS) * 100, 1)
        regime_stats.loc[regime_name, 'Worst Day (%)'] = round(ret_in_regime.min() * 100, 2)

print("SPY Returns by VIX Regime:")
print(regime_stats.to_string())
print("\n→ Crisis regimes: far lower returns AND far higher volatility")
print("  Bull markets account for disproportionately positive returns")

In [ ]:
# ── 2b. Visualize return distributions across regimes ────────────────────────
regime_colors = {
    'Bull (VIX < 15)':    '#27ae60',
    'Normal (15-25)':     '#3498db',
    'Elevated (25-35)':   '#e67e22',
    'Crisis (VIX > 35)':  '#c0392b',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Histogram overlay by regime
for regime_name, color in regime_colors.items():
    mask = regimes == regime_name
    ret_in_regime = spy_aligned[mask]
    if len(ret_in_regime) > 10:
        axes[0].hist(ret_in_regime * 100, bins=50, density=True,
                     alpha=0.5, color=color, label=regime_name)

axes[0].set_title('SPY Return Distribution by Market Regime', fontweight='bold')
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Right: Box plot by regime
regime_returns = [spy_aligned[regimes == rn] * 100
                  for rn in regime_colors if (regimes == rn).sum() > 10]
regime_labels  = [rn for rn in regime_colors if (regimes == rn).sum() > 10]
bp = axes[1].boxplot(regime_returns, patch_artist=True, notch=True,
                     showfliers=True, flierprops=dict(markersize=2, alpha=0.3))

for patch, color in zip(bp['boxes'], [regime_colors[rn] for rn in regime_labels]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1].set_xticklabels([l.replace(' ', '\n') for l in regime_labels], fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Return Spread by Regime (Box Plot)', fontweight='bold')
axes[1].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 2c. Expected value calculation — probability-weighted scenarios ───────────
# Replicating and extending your lecture notes' E[R] formula

print("Replicating E[R] = Σ p_i × R_i from lecture notes:")
print("="*55)

# Use empirical regime frequencies as probabilities
scenarios = regime_stats[['Freq (%)', 'Ann. Return (%)']].copy()
scenarios['Prob'] = scenarios['Freq (%)'] / 100
scenarios['Weighted Return'] = scenarios['Prob'] * scenarios['Ann. Return (%)']

E_R = scenarios['Weighted Return'].sum()

print(scenarios[['Prob', 'Ann. Return (%)', 'Weighted Return']].to_string())
print("-"*55)
print(f"E[R] = Σ p_i × R_i = {E_R:.2f}%")
print(f"\nComparison — simple historical mean: {spy_aligned.mean()*TRADING_DAYS*100:.2f}% annualized")
print("\n→ Probability-weighting by regime gives a more nuanced forward-looking E[R]")

---
## 3. Expected Shortfall (CVaR) — Beyond the VaR Threshold

### Theory

Recall from Module 1.3 that **Value at Risk (VaR)** at 99% confidence answers:
> *"What is the most I will lose on 99% of days?"*

But VaR says nothing about what happens on the remaining 1% of days. Two portfolios can have identical 99% VaR but very different tail behavior — one might lose 1.1× VaR on bad days, the other might lose 5× VaR.

**Expected Shortfall (ES)**, also called **Conditional VaR (CVaR)**, addresses this:

$$
\text{ES}_{\alpha} = E\left[R \mid R \leq \text{VaR}_{\alpha}\right] = \frac{1}{\alpha} \int_0^{\alpha} q_p \, dp
$$

In plain English: **the average loss on the worst $\alpha\%$ of days**.

**Why ES > VaR in practice:**
- **Coherent risk measure** — unlike VaR, ES satisfies all four axioms of a coherent risk measure (including subadditivity: the risk of two assets combined should be ≤ the sum of individual risks)
- **Bank regulation** — Basel III/IV moved from VaR to ES (97.5%) for internal market risk models
- **Portfolio management** — ES better captures the catastrophic scenario risk that keeps risk managers up at night

In [ ]:
# ── 3a. Compute and compare VaR vs CVaR ──────────────────────────────────────
POSITION = 100_000  # $100,000 position

def risk_metrics(returns, name, position=POSITION):
    """Compute VaR and CVaR at multiple confidence levels."""
    results = {'Asset': name}
    for alpha, label in [(0.05, '95%'), (0.01, '99%')]:
        # VaR (quantile)
        var = np.percentile(returns, alpha * 100)
        # CVaR (mean of tail)
        cvar = returns[returns <= var].mean()
        results[f'VaR {label} (%)']  = round(var * 100, 3)
        results[f'CVaR {label} (%)'] = round(cvar * 100, 3)
        results[f'VaR {label} $']    = round(-var * position, 0)
        results[f'CVaR {label} $']   = round(-cvar * position, 0)
    return results

risk_df = pd.DataFrame([
    risk_metrics(spy_ret,  'SPY'),
    risk_metrics(aapl_ret, 'AAPL'),
    risk_metrics(gld_ret,  'GLD'),
]).set_index('Asset')

# Split into % and $ tables for clarity
pct_cols = [c for c in risk_df.columns if '%' in c]
usd_cols = [c for c in risk_df.columns if '$' in c]

print(f"Risk Metrics (% terms):")
print(risk_df[pct_cols].to_string())
print(f"\nRisk Metrics ($ terms — ${POSITION:,} position):")
print(risk_df[usd_cols].to_string())
print("\n→ CVaR is always WORSE than VaR — it tells you the average loss in the tail")

In [ ]:
# ── 3b. Visual: VaR vs CVaR on the return distribution ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, ret, color) in zip(axes, [('SPY', spy_ret, '#2980b9'),
                                          ('AAPL', aapl_ret, '#27ae60'),
                                          ('GLD', gld_ret, '#f39c12')]):
    var99  = np.percentile(ret, 1)
    cvar99 = ret[ret <= var99].mean()

    n, bins, patches = ax.hist(ret * 100, bins=100, density=True,
                                color=color, alpha=0.5, edgecolor='none')

    # Color the tail red
    for patch, left_edge in zip(patches, bins[:-1]):
        if left_edge < var99 * 100:
            patch.set_facecolor('#e74c3c')
            patch.set_alpha(0.8)

    ax.axvline(var99 * 100, color='orange', linewidth=2, linestyle='--',
               label=f'VaR 99%: {var99*100:.2f}%')
    ax.axvline(cvar99 * 100, color='red', linewidth=2, linestyle='-',
               label=f'CVaR 99%: {cvar99*100:.2f}%')

    ax.set_title(f'{name}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Density' if ax == axes[0] else '')
    ax.legend(fontsize=8)

plt.suptitle('VaR vs CVaR — The Tail Beyond the Threshold (1% worst days shaded)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Monte Carlo Simulation — Projecting Future Returns

### Theory

**Monte Carlo simulation** uses random sampling to model the range of possible future outcomes. For stock prices, the most common model is **Geometric Brownian Motion (GBM)**:

$$
S_t = S_0 \cdot \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma \sqrt{t} \cdot Z\right)
$$

Where $Z \sim N(0,1)$ is a standard normal random variable, $\mu$ is the drift (annualized return), and $\sigma$ is the volatility.

At the **daily level**, this simplifies to:
$$
\ln\left(\frac{S_t}{S_{t-1}}\right) = \underbrace{\left(\mu - \frac{\sigma^2}{2}\right) \cdot \Delta t}_{\text{drift}} + \underbrace{\sigma \sqrt{\Delta t} \cdot Z_t}_{\text{random shock}}
$$

**The $\sigma^2/2$ correction** (Itô correction) is important: it accounts for the asymmetry between arithmetic and geometric returns (which you already know from Module 1.1). Without it, you overestimate the expected price level.

### Trading applications of Monte Carlo
- **Options pricing** — simulating many price paths to estimate option payoff distributions
- **Portfolio projections** — showing clients a range of outcomes rather than a single point estimate
- **Risk analysis** — estimating the probability of ruin, reaching a target, or violating a risk limit

In [ ]:
# ── 4a. Monte Carlo simulation for SPY ───────────────────────────────────────
N_SIMS    = 500        # number of simulated paths
N_DAYS    = TRADING_DAYS  # 1 year horizon
INITIAL   = 100        # starting value

# Parameters from historical data
mu_daily    = spy_log.mean()      # historical drift
sigma_daily = spy_log.std()       # historical volatility

# Simulate log returns
drift     = mu_daily - 0.5 * sigma_daily**2   # Itô correction
shocks    = np.random.normal(0, sigma_daily, (N_SIMS, N_DAYS))
daily_log = drift + shocks

# Cumulative product → price paths
price_paths = INITIAL * np.exp(np.cumsum(daily_log, axis=1))

# Final values
final_values = price_paths[:, -1]
final_returns = (final_values / INITIAL - 1) * 100

print(f"Monte Carlo Simulation: {N_SIMS} paths × {N_DAYS} days")
print(f"\nHistorical parameters used:")
print(f"  Daily drift  μ = {mu_daily*100:.4f}%  (annualized: {mu_daily*TRADING_DAYS*100:.2f}%)")
print(f"  Daily vol    σ = {sigma_daily*100:.4f}%  (annualized: {sigma_daily*np.sqrt(TRADING_DAYS)*100:.2f}%)")
print(f"\nSimulated 1-year outcomes:")
print(f"  Median return      : {np.median(final_returns):.2f}%")
print(f"  Mean return        : {np.mean(final_returns):.2f}%")
print(f"  5th percentile     : {np.percentile(final_returns, 5):.2f}%")
print(f"  95th percentile    : {np.percentile(final_returns, 95):.2f}%")
print(f"  Prob of loss (<0%) : {(final_returns < 0).mean()*100:.1f}%")

In [ ]:
# ── 4b. Plot simulated price paths with fan chart ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t = np.arange(N_DAYS)

# Left: Fan chart of price paths
# Plot a subset of paths
for i in range(min(200, N_SIMS)):
    alpha = 0.03 if N_SIMS > 100 else 0.1
    axes[0].plot(t, price_paths[i], color='steelblue', linewidth=0.5, alpha=alpha)

# Percentile bands
p5  = np.percentile(price_paths, 5, axis=0)
p25 = np.percentile(price_paths, 25, axis=0)
p50 = np.percentile(price_paths, 50, axis=0)
p75 = np.percentile(price_paths, 75, axis=0)
p95 = np.percentile(price_paths, 95, axis=0)

axes[0].fill_between(t, p5, p95, alpha=0.15, color='steelblue', label='5th–95th percentile')
axes[0].fill_between(t, p25, p75, alpha=0.25, color='steelblue', label='25th–75th percentile')
axes[0].plot(t, p50, color='navy', linewidth=2.5, label=f'Median path')
axes[0].axhline(INITIAL, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title(f'Monte Carlo: {N_SIMS} Simulated Price Paths (1 Year)',
                  fontweight='bold')
axes[0].set_xlabel('Trading Day')
axes[0].set_ylabel('Portfolio Value (Start = 100)')
axes[0].legend(fontsize=9)

# Right: Distribution of final values
axes[1].hist(final_returns, bins=60, color='steelblue', alpha=0.6, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1.5, linestyle='--', label='Break-even')
axes[1].axvline(np.percentile(final_returns, 5), color='orange', linewidth=2,
                linestyle='--', label=f'5th pct: {np.percentile(final_returns,5):.1f}%')
axes[1].axvline(np.median(final_returns), color='green', linewidth=2,
                linestyle='--', label=f'Median: {np.median(final_returns):.1f}%')

# Shade loss region
loss_data = final_returns[final_returns < 0]
loss_hist, loss_bins = np.histogram(loss_data, bins=30)
axes[1].hist(loss_data, bins=30, color='red', alpha=0.4,
             label=f'Loss region ({(final_returns<0).mean()*100:.0f}% of paths)')

axes[1].set_title('Distribution of 1-Year Returns Across Simulations',
                  fontweight='bold')
axes[1].set_xlabel('1-Year Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Stress Testing — What Would a 2008-Style Crash Do?

### Theory

**Stress testing** applies historically observed extreme scenarios to a current portfolio to estimate potential losses. Unlike Monte Carlo (which uses random sampling from a distribution), stress tests use **actual historical data** from specific crisis episodes.

This is mandatory in institutional finance:
- Banks must stress-test against scenarios like the 2008 GFC, the 1987 crash, etc.
- The Basel III framework requires banks to hold capital sufficient to survive a 10-day 99% VaR scenario
- Portfolio managers use stress tests to identify hidden vulnerabilities

**Key historical crisis episodes for US equities:**

| Event | Period | S&P 500 drawdown |
|---|---|---|
| Dot-com bust | 2000–2002 | ~-49% |
| Global Financial Crisis | Sep 2008 – Mar 2009 | ~-57% |
| COVID crash | Feb–Mar 2020 | ~-34% |
| 2022 bear market | Jan–Oct 2022 | ~-25% |

In [ ]:
# ── 5a. Define and analyze historical stress periods ─────────────────────────
stress_periods = {
    'GFC (Sep 2008 – Mar 2009)': ('2008-09-01', '2009-03-31'),
    'COVID Crash (Feb–Mar 2020)': ('2020-02-15', '2020-03-23'),
    '2022 Bear Market': ('2022-01-01', '2022-10-12'),
    'Rate Shock Q4 2018': ('2018-10-01', '2018-12-24'),
}

spy_price = raw['SPY']
aapl_price = raw['AAPL']
gld_price = raw['GLD']

print("Historical Stress Test — Peak-to-Trough Analysis:")
print(f"{'='*75}")
print(f"{'Period':<30} {'SPY':>8} {'AAPL':>8} {'GLD':>8} {'SPY Max 1D':>12}")
print("-"*75)

for period_name, (start_s, end_s) in stress_periods.items():
    s, e = pd.to_datetime(start_s), pd.to_datetime(end_s)
    mask = (spy_price.index >= s) & (spy_price.index <= e)

    if mask.sum() < 5:
        continue

    spy_chg  = (spy_price[mask].iloc[-1]  / spy_price[mask].iloc[0]  - 1) * 100
    aapl_chg = (aapl_price[mask].iloc[-1] / aapl_price[mask].iloc[0] - 1) * 100
    gld_chg  = (gld_price[mask].iloc[-1]  / gld_price[mask].iloc[0]  - 1) * 100
    spy_worst_day = spy_ret[spy_ret.index.isin(spy_price.index[mask])].min() * 100

    print(f"{period_name:<30} {spy_chg:>7.1f}%  {aapl_chg:>7.1f}%  "
          f"{gld_chg:>7.1f}%  {spy_worst_day:>10.2f}%")

print(f"{'='*75}")
print("\n→ Gold (GLD) acts as a hedge in some crises (GFC) but not all (2022)")
print("→ Understanding asset behavior across SPECIFIC historical scenarios is")
print("  more informative than any statistical model alone")

In [ ]:
# ── 5b. Scenario P&L for a sample portfolio ──────────────────────────────────
# Suppose you have a $50,000 portfolio: 70% SPY + 20% AAPL + 10% GLD

PORTFOLIO_SIZE = 50_000
weights = {'SPY': 0.70, 'AAPL': 0.20, 'GLD': 0.10}
prices_dict = {'SPY': spy_price, 'AAPL': aapl_price, 'GLD': gld_price}

print(f"Portfolio Stress Test: ${PORTFOLIO_SIZE:,} — Weights: {weights}")
print(f"{'='*70}")
print(f"{'Scenario':<30} {'Portfolio Ret':>14} {'Dollar P&L':>12}")
print("-"*70)

for period_name, (start_s, end_s) in stress_periods.items():
    s, e = pd.to_datetime(start_s), pd.to_datetime(end_s)

    port_ret = 0
    for ticker, weight in weights.items():
        price_series = prices_dict[ticker]
        mask = (price_series.index >= s) & (price_series.index <= e)
        if mask.sum() < 5:
            continue
        asset_ret = price_series[mask].iloc[-1] / price_series[mask].iloc[0] - 1
        port_ret += weight * asset_ret

    dollar_pnl = port_ret * PORTFOLIO_SIZE
    print(f"{period_name:<30} {port_ret*100:>13.2f}%  ${dollar_pnl:>10,.0f}")

print(f"{'='*70}")
print("\n→ Even a well-diversified portfolio can lose substantial capital in a crisis")
print("  This is why position sizing and stop-loss disciplines exist")

---
## 6. Level 1 Capstone — Full Risk Profile of an Asset

Bringing together everything from Modules 1.1–1.4 into a single, comprehensive risk report for any ticker.

In [ ]:
# ── Full risk profile function — use on any ticker ───────────────────────────

def full_risk_profile(ticker, start='2015-01-01', end='2024-12-31'):
    """
    Generate a comprehensive risk profile for any stock ticker.
    Applies all concepts from Level 1 Modules 1.1-1.4.
    """
    print(f"\n{'='*60}")
    print(f"  FULL RISK PROFILE: {ticker}")
    print(f"  Period: {start} → {end}")
    print(f"{'='*60}")

    # Download
    data = yf.download(ticker, start=start, end=end,
                       auto_adjust=True, progress=False)['Close']
    ret  = data.pct_change().dropna()
    log_ret = np.log(data / data.shift(1)).dropna()

    n_yrs = len(ret) / TRADING_DAYS

    # ── MODULE 1.1: Returns
    cagr     = (data.iloc[-1] / data.iloc[0]) ** (1/n_yrs) - 1
    arith    = ret.mean() * TRADING_DAYS
    print(f"\n📈 RETURNS (Module 1.1)")
    print(f"   CAGR (geometric)      : {cagr*100:.2f}%")
    print(f"   Arithmetic ann. return: {arith*100:.2f}%")
    print(f"   Total return          : {(data.iloc[-1]/data.iloc[0]-1)*100:.1f}%")

    # ── MODULE 1.3: Volatility
    ann_vol = ret.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (cagr - 0.04) / ann_vol  # assume 4% risk-free rate
    print(f"\n📉 VOLATILITY (Module 1.3)")
    print(f"   Annualized volatility : {ann_vol*100:.2f}%")
    print(f"   Sharpe Ratio (~4% rf) : {sharpe:.3f}")

    # ── MODULE 1.2: Drawdown
    wealth = (1 + ret).cumprod()
    rolling_max = wealth.cummax()
    drawdown = (wealth - rolling_max) / rolling_max
    mdd = drawdown.min()
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan
    print(f"\n🏔️  DRAWDOWN (Module 1.2)")
    print(f"   Max drawdown          : {mdd*100:.2f}%")
    print(f"   Calmar ratio          : {calmar:.3f}")

    # ── MODULE 1.4: Distribution
    skew = stats.skew(ret)
    kurt = stats.kurtosis(ret)
    var99_hist  = np.percentile(ret, 1)
    cvar99_hist = ret[ret <= var99_hist].mean()
    print(f"\n🎲 DISTRIBUTION (Module 1.4)")
    print(f"   Skewness              : {skew:.3f}  ({'negative=left-skewed' if skew<0 else 'positive=right-skewed'})")
    print(f"   Excess Kurtosis       : {kurt:.3f}  ({'fat tails' if kurt>0 else 'thin tails'})")
    print(f"   Historical VaR 99%    : {var99_hist*100:.3f}%/day")
    print(f"   Historical CVaR 99%   : {cvar99_hist*100:.3f}%/day")
    print(f"   Worst single day      : {ret.min()*100:.2f}%")
    print(f"   Best single day       : {ret.max()*100:.2f}%")
    print(f"{'='*60}")

# Run on three tickers
for t in ['SPY', 'AAPL', 'TSLA']:
    full_risk_profile(t)

---
## 7. Module Summary & Key Takeaways

| Concept | Formula | Trading Application |
|---|---|---|
| Skewness | $E[(R-\mu)^3]/\sigma^3$ | Negative skew = crash risk; penalizes high-vol strategies |
| Excess Kurtosis | $E[(R-\mu)^4]/\sigma^4 - 3$ | Fat tails = VaR underestimates real risk |
| Scenario E[R] | $\sum p_i R_i$ | Weight outcomes by probability/regime frequency |
| CVaR / ES | $E[R \mid R \leq VaR_\alpha]$ | Better tail risk measure than VaR alone |
| Monte Carlo | GBM: $S_t = S_0 e^{(\mu-\sigma^2/2)t + \sigma\sqrt{t}Z}$ | Range of outcomes; options pricing; ruin probability |
| Stress testing | Historical scenario returns | Identify portfolio vulnerabilities to specific crises |

### 🔑 Three Rules to Carry Into Your Trading

1. **Never assume normality.** Real returns are skewed and fat-tailed. Models built on Gaussian returns will underestimate the frequency and severity of crashes. Use historical distributions (empirical quantiles) wherever possible.

2. **Use CVaR, not just VaR.** VaR tells you where the bad days start; CVaR tells you how bad they get. For risk management and position sizing, CVaR is the more honest metric.

3. **Stress test before you trade.** Before entering any significant position, run the strategy through at least three historical crisis scenarios. If the losses are unacceptable, reduce size or hedge — not after the fact.

---

## 🎓 Level 1 Complete!

You have now covered all four Level 1 modules:

| Module | Topic | Key concepts |
|---|---|---|
| 1.1 | Financial Returns | Simple, log, dividend-adjusted, CAGR, compound interest |
| 1.2 | Asset Classes & Benchmarks | T-bills, bonds, equity indices, risk premium, Sharpe, drawdown |
| 1.3 | Measuring Risk I: Volatility | Variance, std dev, annualization, EWMA, VIX, VaR |
| 1.4 | Measuring Risk II: Distributions | Skewness, kurtosis, scenarios, CVaR, Monte Carlo, stress testing |

### ➡️ Next: Level 2 — Portfolio Theory & Asset Pricing
Module 2.1 will bring together returns and risk in a **portfolio context** — showing how diversification reduces risk through covariance, and deriving the efficient frontier using real-world data.